# ISOM 839 · Session 8 — When the World Stops Being a Straight Line

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

For seven weeks our models were made of straight lines. Real business is curved: risk grows with the *square* of your bet, and revenue rises, peaks, and falls as you raise the price. Tonight you learn the one question that tells you whether a curved problem is easy, then you build a Nobel Prize-winning portfolio model on real stocks.

### 🗺️ Tonight's road map

| Part | What you do | Time |
|---|---|---|
| **0 · Setup** | Run the first cell (installs CVXPY) | 1 min |
| **1 · Bowl or mountains?** | See the one picture that matters tonight | 5 min |
| **2 · Markowitz lab** | Real stock data → the efficient frontier, in 4 steps | 30 min |
| **3 · Pricing** | Find the sweet-spot price in 4 lines | 5 min |
| **Homework #7 · Option B** | The pricing problem, your turn | at home |

**How to use this notebook:** click a cell and press **Shift + Enter** to run it. Run the cells in order, top to bottom. Every code cell is followed by a **✅ What you should see** note so you always know if you are on track.

In [ ]:
# Part 0 · Setup — installs CVXPY (our curved-problem solver) and yfinance (stock prices). About a minute.
%pip install -q cvxpy yfinance
import numpy as np, pandas as pd, cvxpy as cp
import matplotlib.pyplot as plt
print('Ready! cvxpy', cp.__version__)

---
## Part 1 — Bowl or mountains? The one picture that matters tonight

Drop a marble into a **bowl**: wherever you drop it, it rolls to the same bottom. Drop it into a **mountain range**: it gets stuck in whatever valley is nearby, and you never know if a deeper one exists.

* A **convex** problem is a bowl (or, if you are maximizing, an upside-down bowl: a hill). Solvers find the true best answer, fast.
* A **non-convex** problem is a mountain range. Solvers can get stuck.

Good news: tonight's two business problems are both friendly.

| Problem | Goal | Shape | Easy? |
|---|---|---|---|
| Portfolio risk **w'Σw** | minimize | bowl | ✅ |
| Revenue **p·(a − b·p)** | maximize | hill | ✅ |

Run the cell to see a friendly hill next to a nasty mountain range.

In [ ]:
p = np.linspace(0, 80, 200)
a, b = 1200, 15                                  # demand(p) = a - b*p
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

ax[0].plot(p, p * (a - b * p), lw=3, color='tab:orange')
ax[0].axvline(a / (2 * b), ls='--', c='gray')
ax[0].set_title('FRIENDLY: revenue is a hill with one top'); ax[0].set_xlabel('price ($)'); ax[0].set_ylabel('revenue ($)')

x = np.linspace(-2.4, 2.4, 200)
ax[1].plot(x, x**4 - 4 * x**2 + x, lw=3, color='tab:red')
ax[1].set_title('NASTY: two valleys — which one is the real bottom?'); ax[1].set_xlabel('x')
plt.tight_layout(); plt.show()

**✅ What you should see:** on the left, one smooth hill with its top at **$40** (dashed line). On the right, a curve with two valleys. A solver started near the right valley would stop there, even though the left valley is deeper.

---
## Part 2 — Markowitz on real data (tonight's lab)

**The business question:** how should I split my money across five stocks so the ride is as calm as possible, while still earning my target return?

| Piece | In words | In math |
|---|---|---|
| Decide | What share of my money goes in each stock? | weights **w** |
| Goal | Make the swings as small as possible | minimize **w'Σw** |
| Rules | Hit my target return · invest it all · no short selling | **μ'w ≥ target**, **Σw = 1**, **w ≥ 0** |

The rules are straight lines. Only the goal is curved, and it is a bowl. So it is easy.

### Step 1 — Get real data
Two years of daily prices for five stocks from Yahoo Finance. (If the download is blocked, the cell quietly uses numbers saved from an earlier run, so the lab still works.)

In [ ]:
tickers = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ']
try:
    import yfinance as yf
    prices = yf.download(tickers, period='2y', auto_adjust=True, progress=False)['Close'][tickers]
    rets = prices.pct_change().dropna()
    mu = rets.mean().values * 252                       # annualized expected return
    Sigma = rets.cov().values * 252                     # annualized covariance
    print(f'{len(rets)} trading days downloaded')
except Exception as e:
    print('Download unavailable — using saved statistics. Reason:', type(e).__name__)
    mu = np.array([0.22, 0.25, 0.18, 0.06, 0.04])
    vols = np.array([0.26, 0.24, 0.22, 0.24, 0.15])
    corr = np.array([[1.00, 0.65, 0.35, 0.15, 0.25],
                     [0.65, 1.00, 0.40, 0.15, 0.25],
                     [0.35, 0.40, 1.00, 0.30, 0.30],
                     [0.15, 0.15, 0.30, 1.00, 0.20],
                     [0.25, 0.25, 0.30, 0.20, 1.00]])
    Sigma = np.outer(vols, vols) * corr
n = len(tickers)
print(pd.DataFrame({'exp. return': mu, 'volatility': np.sqrt(np.diag(Sigma))}, index=tickers).round(3))
print('\nCorrelations:'); print(pd.DataFrame(Sigma / np.outer(np.sqrt(np.diag(Sigma)), np.sqrt(np.diag(Sigma))), index=tickers, columns=tickers).round(2))

**✅ What you should see:** two tables. The first lists each stock's yearly **expected return** and **volatility** (how much it swings). The second is the **correlation table**: 1.0 means two stocks move in lockstep, 0 means unrelated.

### Step 2 — Read the correlation table like an investor
Find the **lowest** numbers off the diagonal. Those pairs are like an umbrella shop and a sunscreen shop: when one has a bad day, the other often does not. That is free calm, and it is where all the value of a portfolio comes from. (With live data, your numbers will differ a little from your neighbor's.)

### Step 3 — Solve the model 30 times
Each target return gives one portfolio. Solving for 30 targets traces the whole curve. CVXPY checks that the goal really is a bowl before it solves. If you ever write a mountain by accident, it refuses, and that refusal is a feature.

In [ ]:
w = cp.Variable(n)                                   # DECIDE: share of money in each stock
target = cp.Parameter(nonneg=True)                   # the return we demand (we will change it 30 times)
problem = cp.Problem(
    cp.Minimize(cp.quad_form(w, Sigma)),             # GOAL: smallest swings  (w'Σw)
    [mu @ w >= target,                               # RULE: earn at least the target
     cp.sum(w) == 1,                                 # RULE: invest all the money
     w >= 0])                                        # RULE: no short selling

frontier, weights = [], []
for r in np.linspace(mu.min(), mu.max(), 30):        # 30 targets, from the lowest stock return to the highest
    target.value = r
    problem.solve()
    if problem.status == 'optimal':
        frontier.append((np.sqrt(problem.value), float(mu @ w.value)))   # (risk, ACTUAL return)
        weights.append(w.value.copy())
risk, ret = zip(*frontier)
i_min = int(np.argmin(risk))
print(f'Calmest portfolio: volatility {risk[i_min]:.1%}, expected return {ret[i_min]:.1%}')
print('  weights:', {t: f'{max(float(x), 0):.0%}' for t, x in zip(tickers, weights[i_min])})

**✅ What you should see:** the calmest portfolio's volatility and its weights. With the saved numbers it is about **12.9% volatility and 8.8% return, roughly 58% in JNJ.** Notice: that is *calmer than JNJ alone* (15%), and it earns *more*. Mixing beat the safest single stock on both counts.

### Step 4 — Plot the efficient frontier

In [ ]:
plt.figure(figsize=(8, 4.8))
plt.plot(risk, ret, lw=3, color='tab:orange', label='efficient frontier (best possible mixes)')
for i, t in enumerate(tickers):
    plt.scatter(np.sqrt(Sigma[i, i]), mu[i], s=60, color='tab:blue')
    plt.annotate(t, (np.sqrt(Sigma[i, i]), mu[i]), xytext=(6, 3), textcoords='offset points')
plt.scatter(risk[i_min], ret[i_min], s=140, color='tab:green', zorder=5, label='calmest mix')
plt.xlabel('risk: volatility (yearly)'); plt.ylabel('expected return (yearly)')
plt.title('Every single stock sits below the curve')
plt.legend(); plt.grid(alpha=.3); plt.show()

**✅ What you should see:** an orange curve bending up and to the right, a green dot at its left tip, and five blue dots, all **below or to the right of** the curve.

**Read it like a portfolio manager.** Pick any stock. Look straight up from it to the curve. That vertical gap is extra return you get *for the same risk*, just by mixing. With the saved numbers, at JNJ's risk level the curve earns about **16%** instead of JNJ's **4%**. That gap is diversification, measured.

### 🔧 Your turn (2 minutes)
Real funds cap how much they put in any one stock. Run the next cell: it adds the rule **no stock above 30%** and re-draws the curve. What happened to the curve, and why?

In [ ]:
cap = 0.30                                            # try 0.25 or 0.40 too
capped = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)),
                    [mu @ w >= target, cp.sum(w) == 1, w >= 0, w <= cap])   # one new straight-line rule
risk_c, ret_c = [], []
for r in np.linspace(mu.min(), mu.max(), 30):
    target.value = r
    capped.solve()
    if capped.status == 'optimal':
        risk_c.append(np.sqrt(capped.value)); ret_c.append(float(mu @ w.value))

plt.figure(figsize=(8, 4.8))
plt.plot(risk, ret, lw=3, color='tab:orange', label='no cap')
plt.plot(risk_c, ret_c, lw=3, ls='--', color='tab:purple', label=f'max {cap:.0%} per stock')
plt.xlabel('risk: volatility (yearly)'); plt.ylabel('expected return (yearly)')
plt.legend(); plt.grid(alpha=.3); plt.show()
print(f'Highest return still reachable with the cap: {max(ret_c):.1%}  (without the cap: {max(ret):.1%})')

**✅ What you should see:** the purple curve **starts later and stops short**. It starts later because you can no longer put 58% in calm JNJ, and it stops short because you can no longer go all in on high-return MSFT. A rule can only take options away, so the best curve can only get worse or stay the same. And it is still a bowl: the cap is a straight-line rule.

### 💬 Discussion
Imagine you manage $10B. Which other rules would you add before trusting these weights: sector caps, limits on trading, ESG screens? Which of them are straight lines? (Most are, so it stays a bowl.) Which one would break the bowl? *Hint: "hold at most 10 stocks" is a yes/no rule. Hello again, Session 6.*

---
## Part 3 — Pricing: the sweet-spot price

Charge too little and you leave money on the table. Charge too much and buyers walk away. Somewhere in between is the top of the hill.

* Demand falls as price rises: **demand(p) = a − b·p** (here a = 1,200 and b = 15).
* Revenue = price × demand: **R(p) = p·(a − b·p)**, a hill.
* The top of the hill is where it is flat: **p\* = a / 2b**.

Here it is in CVXPY. We write the revenue as `a*p − b*p²` using `cp.square`, so CVXPY can see it is a hill.

In [ ]:
a, b = 1200, 15
price = cp.Variable(nonneg=True)                                  # DECIDE: the price
rev = cp.Problem(cp.Maximize(a * price - b * cp.square(price)),   # GOAL: biggest revenue (a hill)
                 [price <= a / b])                                # RULE: demand cannot go below zero
rev.solve()
print(f'Best price: ${price.value:.2f}   (formula a/2b = ${a / (2 * b):.2f})   weekly revenue ${rev.value:,.0f}')

**✅ What you should see:** best price **$40.00**, matching the formula, and weekly revenue **$24,000**. The solver and the calculus agree. That is the check you will repeat in the homework.

---
# Homework #7 · Option B — Revenue-Maximizing Pricing 💲

**Pick one of the two Homework #7 options on Canvas: this one, or the inventory MDP in the Session 10 notebook. Same points, same rubric.** Due date on Canvas.

Your product's demand curve, estimated from two years of sales: **a = 1,200, b = 15** (units per week, price in dollars). Unit cost **c = $12**.

1. **Revenue.** Formulate R(p) = p·(a − b·p). Solve for the optimal price **analytically and with CVXPY**, and confirm they agree.
2. **Profit.** Add the unit cost and maximize **(p − c)·(a − b·p)** instead. Solve again, both ways. Which direction did the price move, and why does cost *raise* the optimal price?
3. **Plot** the revenue curve and the profit curve on the same axes with both optima marked.
4. **One paragraph for a manager:** how does the optimal price change as **b** (price sensitivity) rises — try b = 10, 15, 20 — and what does that mean for how you set prices in a market that is becoming more competitive?

In [ ]:
a, b, c = 1200, 15, 12

# TODO 1: revenue-maximizing price, analytic and CVXPY

# TODO 2: profit-maximizing price, analytic and CVXPY  (hint: d/dp of (p-c)(a-bp) = 0)

# TODO 3: plot revenue and profit against price with both optima marked

# TODO 4: sweep b in [10, 15, 20] and print p* for each; write the paragraph in a markdown cell below

---
### Submission checklist (Option B)
- [ ] Both optima computed two ways, and they agree
- [ ] A sentence on why the profit-maximizing price is above the revenue-maximizing one
- [ ] The plot with both optima marked
- [ ] The manager paragraph — pasted into the Canvas text box, starting with "Option B"

**Next week — Session 9:** Markov chains, and the midterm case is due — briefings in class. 📈